In [2]:
import json
import os
from urllib.request import urlopen

from pyspark.sql import SparkSession, functions as F

os.environ.pop("PYSPARK_PYTHON", None)
os.environ.pop("PYSPARK_DRIVER_PYTHON", None)

spark = (
    SparkSession.builder
    .appName("retail-analytics-notebook")
    .master("spark://spark-master:7077")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.autoBroadcastJoinThreshold", "-1")
    .getOrCreate()
)

spark.sparkContext.uiWebUrl

'http://29d79d5cdb27:4040'

In [3]:
left_df = (
    spark.range(0, 100_000)
    .withColumn("group_id", F.col("id") % 20)
    .withColumn("left_value", F.col("id") * 3)
)

right_df = (
    spark.range(0, 100_000)
    .withColumn("right_value", F.col("id") + 7)
)

summary_df = (
    left_df.join(right_df, "id")
    .withColumn("score", F.col("left_value") - F.col("right_value"))
    .groupBy("group_id")
    .agg(
        F.count("*").alias("row_count"),
        F.avg("score").alias("avg_score"),
        F.max("right_value").alias("max_right_value"),
    )
    .orderBy("group_id")
)

summary_df

DataFrame[group_id: bigint, row_count: bigint, avg_score: double, max_right_value: bigint]

In [4]:
summary_rows = summary_df.collect()
summary_rows[:5]

[Row(group_id=0, row_count=5000, avg_score=99973.0, max_right_value=99987),
 Row(group_id=1, row_count=5000, avg_score=99975.0, max_right_value=99988),
 Row(group_id=2, row_count=5000, avg_score=99977.0, max_right_value=99989),
 Row(group_id=3, row_count=5000, avg_score=99979.0, max_right_value=99990),
 Row(group_id=4, row_count=5000, avg_score=99981.0, max_right_value=99991)]

In [5]:
def read_json(url):
    with urlopen(url) as response:
        return json.loads(response.read().decode("utf-8"))

app_id = spark.sparkContext.applicationId
ui_url = spark.sparkContext.uiWebUrl.rstrip("/")
base_url = f"{ui_url}/api/v1/applications/{app_id}"

jobs = read_json(f"{base_url}/jobs")
stages = read_json(f"{base_url}/stages")

job_stats = [
    {
        "job_id": job["jobId"],
        "status": job["status"],
        "num_tasks": job["numTasks"],
        "completed_tasks": job["numCompletedTasks"],
        "failed_tasks": job["numFailedTasks"],
        "stage_ids": job["stageIds"],
    }
    for job in jobs
]

stage_stats = [
    {
        "stage_id": stage["stageId"],
        "name": stage["name"],
        "status": stage["status"],
        "num_tasks": stage["numTasks"],
        "input_mb": round(stage.get("inputBytes", 0) / 1024 / 1024, 2),
        "shuffle_read_mb": round(stage.get("shuffleReadBytes", 0) / 1024 / 1024, 2),
        "shuffle_write_mb": round(stage.get("shuffleWriteBytes", 0) / 1024 / 1024, 2),
        "executor_run_time_s": round(stage.get("executorRunTime", 0) / 1000, 2),
    }
    for stage in stages
]

job_stats, sorted(stage_stats, key=lambda row: row["stage_id"])

([{'job_id': 6,
   'status': 'SUCCEEDED',
   'num_tasks': 12,
   'completed_tasks': 1,
   'failed_tasks': 0,
   'stage_ids': [15, 16, 17, 18, 14]},
  {'job_id': 5,
   'status': 'SUCCEEDED',
   'num_tasks': 11,
   'completed_tasks': 1,
   'failed_tasks': 0,
   'stage_ids': [12, 13, 10, 11]},
  {'job_id': 4,
   'status': 'SUCCEEDED',
   'num_tasks': 11,
   'completed_tasks': 1,
   'failed_tasks': 0,
   'stage_ids': [9, 6, 7, 8]},
  {'job_id': 3,
   'status': 'SUCCEEDED',
   'num_tasks': 10,
   'completed_tasks': 2,
   'failed_tasks': 0,
   'stage_ids': [5, 3, 4]},
  {'job_id': 2,
   'status': 'SUCCEEDED',
   'num_tasks': 4,
   'completed_tasks': 4,
   'failed_tasks': 0,
   'stage_ids': [2]},
  {'job_id': 1,
   'status': 'SUCCEEDED',
   'num_tasks': 4,
   'completed_tasks': 4,
   'failed_tasks': 0,
   'stage_ids': [1]},
  {'job_id': 0,
   'status': 'SUCCEEDED',
   'num_tasks': 2,
   'completed_tasks': 2,
   'failed_tasks': 0,
   'stage_ids': [0]}],
 [{'stage_id': 0,
   'name': 'showString

In [7]:
stages

[{'status': 'COMPLETE',
  'stageId': 18,
  'attemptId': 0,
  'numTasks': 1,
  'numActiveTasks': 0,
  'numCompleteTasks': 1,
  'numFailedTasks': 0,
  'numKilledTasks': 0,
  'numCompletedIndices': 1,
  'submissionTime': '2026-04-22T11:54:30.365GMT',
  'firstTaskLaunchedTime': '2026-04-22T11:54:30.381GMT',
  'completionTime': '2026-04-22T11:54:30.454GMT',
  'executorDeserializeTime': 15,
  'executorDeserializeCpuTime': 7644126,
  'executorRunTime': 31,
  'executorCpuTime': 16593124,
  'resultSize': 9832,
  'jvmGcTime': 0,
  'resultSerializationTime': 0,
  'memoryBytesSpilled': 0,
  'diskBytesSpilled': 0,
  'peakExecutionMemory': 8454128,
  'inputBytes': 0,
  'inputRecords': 0,
  'outputBytes': 0,
  'outputRecords': 0,
  'shuffleRemoteBlocksFetched': 0,
  'shuffleLocalBlocksFetched': 1,
  'shuffleFetchWaitTime': 0,
  'shuffleRemoteBytesRead': 0,
  'shuffleRemoteBytesReadToDisk': 0,
  'shuffleLocalBytesRead': 569,
  'shuffleReadBytes': 569,
  'shuffleReadRecords': 20,
  'shuffleCorruptMerge